# BERT Experiments — Colab (GPU T4)

Este notebook roda dois experimentos que requerem GPU:  
- **E3** `bert-embeddings-linear`: BERTimbau frozen + LogisticRegression  
- **E4** `bert-finetune`: fine-tuning completo do BERTimbau

**Antes de executar:**  
1. Em `Editar > Configurações de notebook`, defina `Acelerador de hardware = GPU (T4)`  
2. Coloque o arquivo `corpus.csv` disponível — ver célula 3 abaixo


In [ ]:
# ── Célula 1: Verificar GPU ──────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
if result.returncode == 0:
    print('GPU detectada:', result.stdout.strip())
else:
    print('AVISO: GPU não detectada. Vá em Editar > Configurações de notebook > GPU')

In [ ]:
# ── Célula 2: Clonar repositório ─────────────────────────────────────────────
!git clone https://github.com/alecsmatos1/analise-de-sentimentos-ecommerce-pln.git repo
%cd repo
!git log --oneline -3

In [ ]:
# ── Célula 3: Instalar dependências ──────────────────────────────────────────
!pip install -q transformers datasets scikit-learn pandas numpy torch torchvision \
                accelerate evaluate safetensors python-dotenv

In [ ]:
# ── Célula 4: Upload do corpus ───────────────────────────────────────────────
# ESCOLHA UMA DAS OPÇÕES:

# --- Opção A: Upload manual (arquivo local) ---
from google.colab import files
import shutil, pathlib

print('Selecione o arquivo corpus.csv do seu computador...')
uploaded = files.upload()  # faz upload do arquivo

# Move para o local esperado
corpus_dest = pathlib.Path('v2/data/processed')
corpus_dest.mkdir(parents=True, exist_ok=True)
for fname in uploaded:
    shutil.move(fname, corpus_dest / 'corpus.csv')
print('Corpus salvo em:', corpus_dest / 'corpus.csv')

# --- Opção B: Carregar do Google Drive ---
# from google.colab import drive
# drive.mount('/content/drive')
# shutil.copy('/content/drive/MyDrive/corpus.csv', 'v2/data/processed/corpus.csv')

# Verificar
import pandas as pd
df = pd.read_csv('v2/data/processed/corpus.csv')
print(f'Corpus: {len(df):,} linhas, colunas: {list(df.columns)}')

In [ ]:
# ── Célula 5: Experimento E3 — BERTimbau frozen + Linear ─────────────────────
# Tempo estimado: 15-30 min no GPU T4
import time
t0 = time.time()
!python v2/run_experiment.py --experiment bert-embeddings-linear
print(f'\nTempo total: {(time.time()-t0)/60:.1f} min')

In [ ]:
# ── Célula 6: Experimento E4 — BERTimbau fine-tune ───────────────────────────
# Tempo estimado: 60-120 min no GPU T4 (3 epochs, corpus completo)
import time
t0 = time.time()
!python v2/run_experiment.py --experiment bert-finetune
print(f'\nTempo total: {(time.time()-t0)/60:.1f} min')

In [ ]:
# ── Célula 7: Ver resultados ─────────────────────────────────────────────────
import json, pathlib
outputs_dir = pathlib.Path('v2/outputs')
for f in sorted(outputs_dir.glob('*.json')):
    data = json.loads(f.read_text())
    exp  = data.get('experiment', f.stem)
    f1   = data.get('f1_macro', data.get('result', {}).get('f1_macro', '?'))
    print(f'{exp:30s}  f1_macro={f1}')

In [ ]:
# ── Célula 8: Baixar JSONs de resultado ─────────────────────────────────────
from google.colab import files
import pathlib

for json_file in pathlib.Path('v2/outputs').glob('bert*.json'):
    print(f'Baixando {json_file.name}...')
    files.download(str(json_file))

print('\nCopie os arquivos baixados para v2/outputs/ no seu computador local.')
print('Depois rode: python v2/run_experiment.py --report compare')